# Module 4 - Topic 4: Overfitting & Underfitting
## Live Demo Notebook

**Scenario:** A Nigerian microfinance lender wants to predict loan defaults.
We deliberately train three models:
1. **Underfit** - too simple (`max_depth=1`)
2. **Overfit** - too complex (`max_depth=None`)
3. **Good fit** - just right (chosen via learning curve)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load and split data
df = pd.read_csv('loan_defaults.csv')
X = df[['income', 'debt_ratio', 'num_late_payments']]
y = df['defaulted']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Dataset:', df.shape, '| Default rate:', f"{y.mean():.1%}")
print(f'Training: {len(X_train)} | Test: {len(X_test)}')

---
## Part 1 - Three Models: Underfit, Overfit, Good Fit

In [ ]:
# Three models with different complexity settings
models = {
    'Underfit  (max_depth=1)': DecisionTreeClassifier(max_depth=1,    random_state=42),
    'Good fit  (max_depth=5)': DecisionTreeClassifier(max_depth=5,    random_state=42),
    'Overfit   (max_depth=None)': DecisionTreeClassifier(max_depth=None, random_state=42),
}

print(f'{"Model":<30} {"Train Acc":>12} {"Test Acc":>12} {"Gap":>10}')
print('-' * 70)

for name, m in models.items():
    m.fit(X_train, y_train)
    tr = m.score(X_train, y_train)
    te = m.score(X_test,  y_test)
    gap = tr - te
    flag = '<-- OVERFIT' if gap > 0.15 else ('<-- UNDERFIT' if tr < 0.65 else '<-- GOOD')
    print(f'{name:<30} {tr:>11.1%} {te:>11.1%} {gap:>9.1%}  {flag}')

---
## Part 2 - The Learning Curve

We train a model for every `max_depth` from 1 to 20 and plot training vs test accuracy.

In [ ]:
depths = list(range(1, 21))
train_scores, test_scores = [], []

for depth in depths:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    train_scores.append(m.score(X_train, y_train))
    test_scores.append(m.score(X_test, y_test))

# Find best depth
best_idx   = test_scores.index(max(test_scores))
best_depth = depths[best_idx]
best_acc   = test_scores[best_idx]

# Plot
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(depths, train_scores, label='Training accuracy', color='steelblue',
        marker='o', markersize=5, linewidth=2)
ax.plot(depths, test_scores,  label='Test accuracy',     color='coral',
        marker='o', markersize=5, linewidth=2)

# Mark sweet spot
ax.axvline(best_depth, color='green', linestyle='--', linewidth=1.5,
           label=f'Best depth = {best_depth} (test acc = {best_acc:.1%})')

# Annotate regions
ax.axvspan(1, 2.5, alpha=0.07, color='orange', label='Underfitting zone')
ax.axvspan(10, 20, alpha=0.07, color='red',    label='Overfitting zone')

ax.set_xlabel('max_depth (model complexity →)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Learning Curve: Underfitting → Sweet Spot → Overfitting',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0.4, 1.05)
ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Best max_depth: {best_depth}')
print(f'Best test accuracy: {best_acc:.2%}')

---
## Part 3 - Diagnosing Each Zone

In [ ]:
# UNDERFITTING - depth=1
underfit = DecisionTreeClassifier(max_depth=1, random_state=42)
underfit.fit(X_train, y_train)

tr = underfit.score(X_train, y_train)
te = underfit.score(X_test, y_test)

print('=== UNDERFITTING (max_depth=1) ===')
print(f'Training accuracy: {tr:.2%}')
print(f'Test accuracy:     {te:.2%}')
print(f'Gap:               {(tr-te):.2%}')
print()
print('Diagnosis: both scores are low - the model is too simple.')
print('The tree can only ask ONE question before deciding.')
print('Fix: increase max_depth, add more features, try a more powerful algorithm.')

In [ ]:
# OVERFITTING - no depth limit
overfit = DecisionTreeClassifier(max_depth=None, random_state=42)
overfit.fit(X_train, y_train)

tr = overfit.score(X_train, y_train)
te = overfit.score(X_test, y_test)

print('=== OVERFITTING (max_depth=None) ===')
print(f'Training accuracy: {tr:.2%}')
print(f'Test accuracy:     {te:.2%}')
print(f'Gap:               {(tr-te):.2%}')
print()
print('Diagnosis: near-perfect training accuracy but poor test accuracy.')
print('The model memorised the training data - including noise.')
print('A 100% training accuracy is almost always a red flag.')
print('Fix: reduce max_depth, add min_samples_leaf, collect more data.')

In [ ]:
# GOOD FIT - best depth from learning curve
good_fit = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
good_fit.fit(X_train, y_train)

tr = good_fit.score(X_train, y_train)
te = good_fit.score(X_test, y_test)

print(f'=== GOOD FIT (max_depth={best_depth}) ===')
print(f'Training accuracy: {tr:.2%}')
print(f'Test accuracy:     {te:.2%}')
print(f'Gap:               {(tr-te):.2%}')
print()
print('Diagnosis: training and test accuracy are close, test accuracy is acceptable.')
print('This model generalises - it learned patterns, not specific training examples.')
print('This is the model we would take forward.')

---
## Part 4 - Regularisation: Adding Constraints to Prevent Overfitting

In [ ]:
# Regularised model - add min_samples_leaf to prevent over-specific rules
regularised = DecisionTreeClassifier(
    max_depth=None,        # no depth limit
    min_samples_leaf=10,   # each leaf needs at least 10 training examples
    min_samples_split=20,  # a node needs 20 examples before it can split
    random_state=42
)
regularised.fit(X_train, y_train)

tr = regularised.score(X_train, y_train)
te = regularised.score(X_test, y_test)

print('=== REGULARISED (no depth limit but leaf constraints) ===')
print(f'Training accuracy: {tr:.2%}')
print(f'Test accuracy:     {te:.2%}')
print(f'Gap:               {(tr-te):.2%}')
print()
print('Regularisation forces the tree to create rules that apply to many examples.')
print('Each leaf must contain at least 10 training points - prevents hyper-specific rules.')

---
## Part 5 - Summary Comparison

In [ ]:
# Final comparison table
results = [
    ('Underfit  (depth=1)',    DecisionTreeClassifier(max_depth=1,    random_state=42)),
    ('Overfit   (depth=None)', DecisionTreeClassifier(max_depth=None, random_state=42)),
    (f'Good fit  (depth={best_depth})',  DecisionTreeClassifier(max_depth=best_depth, random_state=42)),
    ('Regularised (leaf>=10)', DecisionTreeClassifier(max_depth=None, min_samples_leaf=10, random_state=42)),
]

print(f'{"Model":<30} {"Train":>8} {"Test":>8} {"Gap":>8} {"Verdict":>12}')
print('-' * 75)
for name, m in results:
    m.fit(X_train, y_train)
    tr = m.score(X_train, y_train)
    te = m.score(X_test,  y_test)
    gap = tr - te
    if tr < 0.7:
        verdict = 'UNDERFIT'
    elif gap > 0.15:
        verdict = 'OVERFIT'
    else:
        verdict = 'GOOD FIT'
    print(f'{name:<30} {tr:>7.1%} {te:>8.1%} {gap:>7.1%} {verdict:>12}')

---
## Summary

**Three failure modes and their signatures:**

| Scenario | Train acc | Test acc | Gap | Diagnosis |
|---|---|---|---|---|
| Underfitting | Low | Low | Small | Model too simple |
| Overfitting | High | Low | Large | Model too complex |
| Good fit | High | High | Small | Just right |

**The learning curve** is your diagnostic tool - plot training vs test accuracy across model complexity.

**The quick checklist:**
- Training accuracy high? ✓
- Test accuracy also high? ✓  
- Gap < 10%? ✓ → Deploy

**Next - Topic 5:** Introduction to scikit-learn. The full API: `fit`, `predict`, `transform`, `Pipeline`.